In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Step 1.1: Load the CSV files
mitbih_train = pd.read_csv('mitbih_train.csv', header=None)
mitbih_test = pd.read_csv('mitbih_test.csv', header=None)

# Step 1.2: Separate features and labels
X_train = mitbih_train.iloc[:, :-1].values
y_train = mitbih_train.iloc[:, -1].values

X_test = mitbih_test.iloc[:, :-1].values
y_test = mitbih_test.iloc[:, -1].values

# Step 1.3: Initial train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Step 1.4: Apply SMOTE to balance the classes in the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Step 1.5: Standardize data
scaler = StandardScaler()
X_train_smote = scaler.fit_transform(X_train_smote)
X_test = scaler.transform(X_test)


In [2]:
from scipy.signal import find_peaks

# Example of feature extraction: peak detection in ECG signals
def extract_features(X):
    features = []
    for ecg in X:
        peaks, _ = find_peaks(ecg, distance=200)  # Example peak detection
        feature = [
            len(peaks),  # Number of peaks
            np.mean(np.diff(peaks)) if len(peaks) > 1 else 0,  # Mean distance between peaks
            np.max(ecg),  # Maximum amplitude
            np.min(ecg)   # Minimum amplitude
        ]
        features.append(feature)
    return np.array(features)

# Extract features
X_train_features = extract_features(X_train_smote)
X_test_features = extract_features(X_test)

# Combine the engineered features with the original data
X_train_combined = np.hstack([X_train_smote, X_train_features])
X_test_combined = np.hstack([X_test, X_test_features])


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, InputLayer

# Step 3.1: Reshape the original data for Conv2D
X_train_reshaped = X_train_smote.reshape(-1, 17, 11, 1)
X_test_reshaped = X_test.reshape(-1, 17, 11, 1)

# Step 3.2: Create the CNN model
def create_cnn_model():
    model = Sequential()
    
    # Convolutional layers
    model.add(InputLayer(input_shape=(17, 11, 1)))
    model.add(Conv2D(32, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))
    
    model.add(Conv2D(64, kernel_size=(2, 2), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))
    
    model.add(Flatten())

    # Add dense layers that incorporate the engineered features
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))
    
    # Output layer
    model.add(Dense(5, activation='softmax'))  # 5 classes for MIT-BIH
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Step 3.3: Combine original CNN model with engineered features
cnn_model = create_cnn_model()


c:\Users\asena\anaconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:25: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [4]:
from sklearn.utils.class_weight import compute_class_weight

# Step 4.1: Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_smote), y=y_train_smote)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

# Step 4.2: Train the CNN model with class weights
history = cnn_model.fit(X_train_reshaped, y_train_smote,
                        epochs=20,  # Increased epochs to allow more training
                        batch_size=256,
                        validation_data=(X_test_reshaped, y_test),
                        class_weight=class_weight_dict,
                        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])


Epoch 1/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - accuracy: 0.6635 - loss: 0.8537 - val_accuracy: 0.8493 - val_loss: 0.5390
Epoch 2/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - accuracy: 0.8662 - loss: 0.3738 - val_accuracy: 0.8932 - val_loss: 0.3752
Epoch 3/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.8912 - loss: 0.3058 - val_accuracy: 0.8966 - val_loss: 0.3198
Epoch 4/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - accuracy: 0.9044 - loss: 0.2716 - val_accuracy: 0.9147 - val_loss: 0.2689
Epoch 5/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.9154 - loss: 0.2441 - val_accuracy: 0.9157 - val_loss: 0.2510
Epoch 6/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9195 - loss: 0.2313 - val_accuracy: 0.9119 - val_loss: 0.2527
Epoch 7/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9230 - loss: 0.2204 - val_accuracy: 0.9147 - val_loss: 0.2394
Epoch 8/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.9275 -

In [5]:
from sklearn.metrics import classification_report

# Step 5.1: Evaluate the model on the test set
test_loss, test_accuracy = cnn_model.evaluate(X_test_reshaped, y_test)
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")

# Step 5.2: Make predictions and generate a classification report
y_pred = np.argmax(cnn_model.predict(X_test_reshaped), axis=-1)
classification_report_str = classification_report(y_test, y_pred, target_names=['N', 'S', 'V', 'F', 'Q'])
print("Classification Report:\n", classification_report_str)


548/548 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9358 - loss: 0.1827
Final Test Accuracy: 93.79%
548/548 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Classification Report:
               precision    recall  f1-score   support

           N       0.99      0.94      0.96     14494
           S       0.46      0.84      0.59       445
           V       0.85      0.94      0.89      1158
           F       0.26      0.86      0.40       128
           Q       0.96      0.99      0.97      1286

    accuracy                           0.94     17511
   macro avg       0.71      0.91      0.77     17511
weighted avg       0.96      0.94      0.95     17511

